# TPNWHT2 — Evaluation Notebook

Two-Pronged Network with WHT Layer **and learnable decimation layers**.  
Loads checkpoints **`TPNWHT2_best_test_model_real_parts.pth`** and **`TPNWHT2_best_test_model_imag_parts.pth`**,  
computes all paper metrics (MACE, CVar, CSD, NAACC, RMSEP25S) and generates figures.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FuncFormatter
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import seaborn as sns
import os

from models import TPNWHT2

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load data and checkpoints

In [ ]:
DATA_DIR = '.'
CKPT_DIR = 'checkpoints'
SR = 1500

x_test      = np.load(os.path.join(DATA_DIR, 'x_test.npy'))
y_test      = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
y_test_real = y_test[:, :16]
y_test_imag = y_test[:, 16:]

x_t    = torch.tensor(x_test, dtype=torch.float32)
loader = DataLoader(TensorDataset(x_t), batch_size=256, shuffle=False)

# ── TPNWHT2: two separate models ──────────────────────────────────────────────
real_model = TPNWHT2().to(device)
imag_model = TPNWHT2().to(device)

real_model.load_state_dict(torch.load(
    os.path.join(CKPT_DIR, 'TPNWHT2_best_test_model_real_parts.pth'), map_location=device))
imag_model.load_state_dict(torch.load(
    os.path.join(CKPT_DIR, 'TPNWHT2_best_test_model_imag_parts.pth'), map_location=device))
real_model.eval(); imag_model.eval()

n_params = sum(p.numel() for p in real_model.parameters())
print(f'TPNWHT2 loaded  |  parameters per branch: {n_params:,}  |  total: {2*n_params:,}')

## 2. Run inference

In [ ]:
real_preds, imag_preds = [], []
with torch.no_grad():
    for (x,) in loader:
        x = x.to(device)
        real_preds.append(real_model(x).cpu().numpy())
        imag_preds.append(imag_model(x).cpu().numpy())

real_pred  = np.concatenate(real_preds)
imag_pred  = np.concatenate(imag_preds)
phase_pred = np.arctan2(imag_pred,  real_pred)
phase_true = np.arctan2(y_test_imag, y_test_real)
print('phase_pred shape:', phase_pred.shape)

## 3. Evaluation metrics

In [ ]:
def mace(true, pred):
    return np.degrees(np.mean(np.abs(np.angle(np.exp(1j*(pred-true))))))

def cvar(true, pred):
    d = np.angle(np.exp(1j*(pred-true)))
    return 1 - np.abs(np.mean(np.exp(1j*d)))

def csd(true, pred):
    d = np.angle(np.exp(1j*(pred-true)))
    return np.degrees(np.sqrt(-2*np.log(np.abs(np.mean(np.exp(1j*d))))))

def naacc(true, pred):
    d = np.degrees(np.abs(np.angle(np.exp(1j*(pred-true)))))
    return 100*(1 - np.mean(d)/180)

def rmsep25s(true, pred, sr=1500):
    d = ((np.degrees(pred)-np.degrees(true))+180)%360-180
    d = d.reshape(-1)
    win = int(sr*0.25)
    return np.mean([np.sqrt(np.mean(d[i*win:(i+1)*win]**2)) for i in range(len(d)//win)])

print(f'MACE     : {mace(phase_true, phase_pred):.4f} °')
print(f'CVar     : {cvar(phase_true, phase_pred):.2e}')
print(f'CSD      : {csd(phase_true, phase_pred):.4f} °')
print(f'NAACC    : {naacc(phase_true, phase_pred):.4f} %')
print(f'RMSEP25S : {rmsep25s(phase_true, phase_pred, SR):.4f} °')

## 4. Phase comparison plot

In [ ]:
def pi_fmt(v, _):
    m = {0:'0', np.pi:r'$\pi$', -np.pi:r'$-\pi$',
         np.pi/2:r'$\pi/2$', -np.pi/2:r'$-\pi/2$'}
    return m.get(round(v,6), f'{v:.2f}')

p_true  = phase_true.reshape(-1)
p_pred  = phase_pred.reshape(-1)
t       = np.arange(len(p_true)) / SR
seg     = slice(0, min(int(2*SR), len(p_true)))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[seg], p_true[seg], color='#E26472', label='True Phase', lw=2, alpha=0.8)
ax.plot(t[seg], p_pred[seg], color='#2B9CD7', label='Predicted Phase TPNWHT2',
        lw=2, ls='--', alpha=0.8)
ax.set_xlabel('Time (seconds)', fontsize=14)
ax.set_ylabel('Phase (radians)', fontsize=14)
ax.yaxis.set_major_locator(MultipleLocator(np.pi/2))
ax.yaxis.set_major_formatter(FuncFormatter(pi_fmt))
ax.legend(fontsize=12); ax.grid(False)
plt.tight_layout()
plt.savefig('TPNWHT2_phase_comparison.svg', format='svg', bbox_inches='tight')
plt.show()

## 5. Rose diagrams (0°, 90°, 180°, 270°)

In [ ]:
def rose_diagram(true_phases, pred_phases, target_deg=None, num_bins=180, title=''):
    p_t    = true_phases % (2*np.pi)
    errors = np.angle(np.exp(1j*(pred_phases - true_phases)))
    if target_deg is not None:
        tr   = np.radians(target_deg)
        mask = np.abs(np.angle(np.exp(1j*(p_t-tr)))) < np.radians(5)
        p_t, errors = p_t[mask], errors[mask]
    bins   = np.linspace(0, 2*np.pi, num_bins+1)
    ctrs   = (bins[:-1]+bins[1:])/2
    w      = 2*np.pi/num_bins
    idx    = np.digitize(p_t, bins)-1
    binn   = [np.mean(np.abs(errors[idx==i])) if np.any(idx==i) else 0 for i in range(num_bins)]
    me     = np.degrees(np.mean(np.abs(errors)))
    se     = np.degrees(np.std(np.abs(errors)))
    fig,ax = plt.subplots(subplot_kw={'projection':'polar'}, figsize=(5,5))
    ax.bar(ctrs, binn, width=w, color='none', edgecolor='#9d84bf', linewidth=1.2, alpha=0.9)
    ax.set_title(f'{title}\nMean Error: {me:.2f}°, Std: {se:.2f}°', va='bottom', fontsize=11)
    plt.tight_layout(); plt.show()

pf_true = phase_true.reshape(-1)
pf_pred = phase_pred.reshape(-1)
for deg in [0, 90, 180, 270]:
    rose_diagram(pf_true, pf_pred, target_deg=deg, title=f'TPNWHT2 — Phase: {deg}°')